In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Compute ensemble-mean EP flux and divergence for February and March, both coupled and nocoupled experiments for year 0008.
Generate for each dataset:
1. Height–Pressure Profile
2. Time-mean EP flux & divergence
3. EP flux timeseries GIF
Outputs saved under `/home/weiji/restart_exam/plots/epflux/{dataset_tag}`.
"""

import os
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import imageio
from metpy.calc import potential_temperature
from metpy.units import units

# --- Constants ---
a = 6.37122e6         # Earth radius (m)
Omega = 7.2921e-5      # Earth rotation rate (rad/s)
g = 9.806             # Gravity (m/s^2)
p0 = 1e5             # Reference pressure (Pa)
H = 7000.0            # Scale height (m)


def compute_ep_flux_divergence(ds):
    """
    Compute Eliassen-Palm (EP) flux components and divergence for a single dataset.
    Returns DataArrays Fphi, Fz, EP_div, plus lat (deg) and z (m).
    """
    U = ds['U'] #m/s
    V = ds['V'] #m/s
    T = ds['T'] #K
    plev = ds['plev'] #pa
    lat = ds['lat']

    # potential temperature
    theta = potential_temperature(plev * units.hPa, T * units.K)

    # zonal mean and eddy
    u_zm = U.mean(dim='lon') #m/s
    v_zm = V.mean(dim='lon')#m/s
    theta_zm = theta.mean(dim='lon')#k
    u_za = U - u_zm#m/s
    v_za = V - v_zm#m/s
    theta_za = theta - theta_zm#k

    # eddy covariances
    UVzm = (u_za * v_za).mean(dim='lon')#m^2s^2
    VTHETAzm = (v_za * theta_za).mean(dim='lon')#m*k/s

    # vertical coordinate (height)
    z = -H * np.log(plev / p0)#m
    rho0_1d = p0 / (g * H) * np.exp(-z / H) #pa*s^2/m^2
    rho0 = xr.DataArray(rho0_1d, coords={'plev': plev}, dims=('plev',)).expand_dims(lat=lat)

    phi = np.deg2rad(lat)
    f = 2 * Omega * np.sin(phi)
    cosphi = np.cos(phi)
    acphi = a * cosphi

    # gradient of mean theta
    theta_zm = theta_zm.assign_coords({'z': z})
    dthetaz = theta_zm.differentiate(coord='z', edge_order=2)

    # EP flux components
    Fphi = - rho0 * acphi * UVzm
    Fz = f * acphi * rho0 * VTHETAzm / dthetaz

    Fphi = (Fphi.assign_coords({'phi': phi})
                .assign_attrs(var_name="meridional EP flux Fφ", units="kg/s²"))
    Fz = (Fz.assign_coords({'z': z})
             .assign_attrs(var_name="vertical EP flux Fz", units="kg/s²"))

    # divergence
    div_phi = ((Fphi * cosphi).differentiate('phi', edge_order=2)) / acphi
    div_z = Fz.differentiate('z', edge_order=2)
    EP_div = (div_phi + div_z) / (rho0 * acphi) * (3600 * 24)
    EP_div = (EP_div.assign_attrs(var_name="EP flux divergence", units="m/s/day")
                     .assign_coords({'z': z}))

    return Fphi, Fz, EP_div, lat, z


def plot_height_pressure(plev, z, output_dir, dataset_tag):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(z / 1000.0, plev, '-o')
    ax.invert_yaxis()
    ax.set_xlabel('Height (km)')
    ax.set_ylabel('Pressure (Pa)')
    ax.set_title(f'Height–Pressure Profile ({dataset_tag})')
    fig.tight_layout()
    path = os.path.join(output_dir, f'height_pressure_profile_{dataset_tag}.png')
    fig.savefig(path)
    plt.close(fig)



'''
# 仅用 10 km 以上点计算归一化因子,因为平流变化大所以只看10km以上部分的矢量。
# 上面注释掉的部分就是绘制所有的矢量的代码。
def plot_time_mean_ep_flux(Fphi, Fz, EP_div, lat, z, output_dir, dataset_tag):
    import os
    import numpy as np
    import matplotlib.pyplot as plt

    # make 2D grids of latitude and height (km)
    lat2d, z2d = np.meshgrid(lat, z/1000.0)
    s_plev, s_lat = 2, 2

    # compute time‐mean fields
    Fphi_m  = Fphi.mean(dim='time')
    Fz_m    = Fz.mean(dim='time')
    EPdiv_m = EP_div.mean(dim='time')

    # plot divergence
    fig, ax = plt.subplots(figsize=(10, 6))
    cf = ax.contourf(
        lat2d, z2d, EPdiv_m.T,
        levels=np.linspace(-5, 5, 41),
        cmap='RdBu_r', extend='both'
    )
    cbar = fig.colorbar(
        cf, ax=ax,
        orientation='vertical',
        pad=0.02, fraction=0.05
    )
    cbar.set_label('Time‐mean EP flux divergence (m/s/day)')
    # white contour lines
    cs = ax.contour(
        lat2d, z2d, EPdiv_m.T,
        levels=np.arange(-5, 6),
        colors='white', linewidths=0.5
    )
    ax.clabel(cs, fmt='%d', inline=True, fontsize=8)

    # prepare full arrays of vectors
    U_full = Fphi_m.values   # shape (plev, lat) or (lat, plev)
    V_full = Fz_m.values
    # ensure first axis is plev, second is lat
    if U_full.shape != lat2d.shape:
        U_full = U_full.T
        V_full = V_full.T

    # mask entire layers below 10 km
    z_km = z / 1000.0
    mask_z = z_km < 10.0
    U_full[mask_z, :] = np.nan
    V_full[mask_z, :] = np.nan

    # subsample for quiver
    X = lat2d[::s_plev, ::s_lat]
    Y = z2d[::s_plev, ::s_lat]
    Uq = U_full[::s_plev, ::s_lat]
    Vq = V_full[::s_plev, ::s_lat]

    # normalize based only on >=10 km
    amp = np.nanmax(np.hypot(Uq, Vq))
    if amp > 0:
        Uq = Uq / amp * 15
        Vq = Vq / amp * 15

    ax.quiver(
        X, Y, Uq, Vq,
        angles='xy', scale_units='xy', scale=1,
        width=0.003, headwidth=4
    )

    ax.set_ylim(0, 21)
    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Height (km)')
    ax.set_title(f'Time‐mean QG EP Flux & Divergence ({dataset_tag})')
    fig.tight_layout()
    path = os.path.join(output_dir, f'time_mean_EPflux_{dataset_tag}.png')
    fig.savefig(path)
    plt.close(fig)


def create_ep_flux_gif(Fphi, Fz, EP_div, lat, z, output_dir, dataset_tag, duration=0.5):
    import os
    import numpy as np
    import matplotlib.pyplot as plt
    import imageio

    lat2d, z2d = np.meshgrid(lat, z/1000.0)
    s_plev, s_lat = 2, 2

    # precompute mask for below-10km layers
    z_km = z / 1000.0
    mask_z = z_km < 10.0

    frame_paths = []
    for i in range(Fphi.sizes['time']):
        EPt = EP_div.isel(time=i)
        fig, ax = plt.subplots(figsize=(10, 6))
        cf = ax.contourf(
            lat2d, z2d, EPt.T,
            levels=np.linspace(-5, 5, 41),
            cmap='RdBu_r', extend='both'
        )
        cbar = fig.colorbar(
            cf, ax=ax,
            orientation='vertical',
            pad=0.02, fraction=0.05
        )
        cbar.set_label('EP flux divergence (m/s/day)')
        cs = ax.contour(
            lat2d, z2d, EPt.T,
            levels=np.arange(-5, 6),
            colors='white', linewidths=0.5
        )
        ax.clabel(cs, fmt='%d', inline=True, fontsize=8)

        # get vector arrays and mask
        U_full = Fphi.isel(time=i).values
        V_full = Fz.isel(time=i).values
        if U_full.shape != lat2d.shape:
            U_full = U_full.T
            V_full = V_full.T
        U_full[mask_z, :] = np.nan
        V_full[mask_z, :] = np.nan

        X = lat2d[::s_plev, ::s_lat]
        Y = z2d[::s_plev, ::s_lat]
        Uq = U_full[::s_plev, ::s_lat]
        Vq = V_full[::s_plev, ::s_lat]

        amp = np.nanmax(np.hypot(Uq, Vq))
        if amp > 0:
            Uq = Uq / amp * 15
            Vq = Vq / amp * 15

        ax.quiver(
            X, Y, Uq, Vq,
            angles='xy', scale_units='xy', scale=1,
            width=0.003, headwidth=4
        )

        ax.set_ylim(0, 21)
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_title(f'EP Flux at {str(Fphi["time"].values[i])} ({dataset_tag})')
        fig.tight_layout()

        frame_file = os.path.join(output_dir, f'frame_{i:03d}_{dataset_tag}.png')
        fig.savefig(frame_file)
        plt.close(fig)
        frame_paths.append(frame_file)

    gif_path = os.path.join(output_dir, f'EP_flux_timeseries_{dataset_tag}.gif')
    with imageio.get_writer(gif_path, mode='I', duration=duration) as writer:
        for fp in frame_paths:
            writer.append_data(imageio.imread(fp))

    return gif_path
'''
def plot_time_mean_ep_flux(Fphi, Fz, EP_div, lat, z, output_dir, dataset_tag):
    lat2d, z2d = np.meshgrid(lat, z / 1000.0)
    s_plev, s_lat = 2, 2
    Fphi_m = Fphi.mean(dim='time')
    Fz_m = Fz.mean(dim='time')
    EPdiv_m = EP_div.mean(dim='time')

    fig, ax = plt.subplots(figsize=(10, 6))
    cf = ax.contourf(lat2d, z2d, EPdiv_m.T,
                     levels=np.linspace(-5, 5, 41), cmap='RdBu_r', extend='both')
    fig.colorbar(cf, ax=ax, label='Time‐mean EP flux divergence (m/s/day)')
    contours = ax.contour(lat2d, z2d, EPdiv_m.T,
                          levels=np.arange(-5, 6, 1), colors='k', linewidths=0.5)
    ax.clabel(contours, fmt='%d', inline=True, fontsize=8)

    X = lat2d[::s_plev, ::s_lat]
    Y = z2d[::s_plev, ::s_lat]
    Uq = Fphi_m.values[::s_plev, ::s_lat]
    Vq = Fz_m.values[::s_plev, ::s_lat]
    Uq_norm = Uq / np.nanmax(np.abs(Uq)) * 5
    Vq_norm = Vq / np.nanmax(np.abs(Vq)) * 5
    ax.quiver(X, Y, Uq_norm, Vq_norm,
              angles='xy', scale_units='xy', scale=1,
              width=0.003, headwidth=4)

    ax.set_ylim(0, 35)
    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Height (km)')
    ax.set_title(f'Time‐mean QG EP Flux & Divergence ({dataset_tag})')
    fig.tight_layout()
    path = os.path.join(output_dir, f'time_mean_EPflux_{dataset_tag}.png')
    fig.savefig(path)
    plt.close(fig)


def create_ep_flux_gif(Fphi, Fz, EP_div, lat, z, output_dir, dataset_tag, duration=0.5):
    lat2d, z2d = np.meshgrid(lat, z / 1000.0)
    s_plev, s_lat = 2, 2
    frame_paths = []
    for i in range(Fphi.sizes['time']):
        EPdiv_t = EP_div.isel(time=i)
        fig, ax = plt.subplots(figsize=(10, 6))
        cf = ax.contourf(lat2d, z2d, EPdiv_t.T,
                         levels=np.linspace(-5, 5, 41), cmap='RdBu_r', extend='both')
        fig.colorbar(cf, ax=ax, label='EP flux divergence (m/s/day)')
        contours = ax.contour(lat2d, z2d, EPdiv_t.T,
                              levels=np.arange(-5, 6, 1), colors='k', linewidths=0.5)
        ax.clabel(contours, fmt='%d', inline=True, fontsize=8)
        X = lat2d[::s_plev, ::s_lat]
        Y = z2d[::s_plev, ::s_lat]
        Uq = Fphi.isel(time=i).values[::s_plev, ::s_lat]
        Vq = Fz.isel(time=i).values[::s_plev, ::s_lat]
        Uq_norm = Uq / np.nanmax(np.abs(Uq)) * 5
        Vq_norm = Vq / np.nanmax(np.abs(Vq)) * 5
        ax.quiver(X, Y, Uq_norm, Vq_norm,
                  angles='xy', scale_units='xy', scale=1,
                  width=0.003, headwidth=4)
        ax.set_ylim(0, 21)
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        time_str = str(Fphi['time'].values[i])
        ax.set_title(f'EP Flux at {time_str} ({dataset_tag})')
        fig.tight_layout()
        frame_file = os.path.join(output_dir, f'frame_{i:03d}_{dataset_tag}.png')
        fig.savefig(frame_file)
        plt.close(fig)
        frame_paths.append(frame_file)

    gif_file = os.path.join(output_dir, f'EP_flux_timeseries_{dataset_tag}.gif')
    with imageio.get_writer(gif_file, mode='I', duration=duration) as writer:
        for f in frame_paths:
            writer.append_data(imageio.imread(f))
    return gif_file







In [ ]:
# --- Main execution and storage ---
year = '0008'
target_dir_parent = "/home/weiji/restart_exam/plots/epflux_speedtest"
scenarios = [
    (f"{year}_Feb_couple",   "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc"),
    (f"{year}_Mar_couple",   "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc"),
    (f"{year}_Feb_nocouple", f"/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc"),
    (f"{year}_Mar_nocouple", f"/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc"),
]

scenario_results = {}  # store (Fphi, Fz, EP_div, lat, z) for each scenario

# Loop over experiments
for idx, (tag, pattern) in enumerate(scenarios):
    out_dir = os.path.join(target_dir_parent, tag)
    os.makedirs(out_dir, exist_ok=True)

    files = sorted(glob.glob(pattern))
    fphi_list, fz_list, ep_list = [], [], []
    for fp in files:
        ds = xr.open_dataset(fp)
        Fphi_i, Fz_i, EP_i, lat_vals, z_vals = compute_ep_flux_divergence(ds)
        fphi_list.append(Fphi_i); fz_list.append(Fz_i); ep_list.append(EP_i)

    Fphi = xr.concat(fphi_list, dim='member').mean(dim='member')
    Fz    = xr.concat(fz_list,   dim='member').mean(dim='member')
    EP_div= xr.concat(ep_list,   dim='member').mean(dim='member')

    # first scenario only: height-pressure
    if idx == 0:
        plot_height_pressure(ds['plev'].values, z_vals, target_dir_parent, tag)

    plot_time_mean_ep_flux(Fphi, Fz, EP_div, lat_vals.values, z_vals, out_dir, tag)
    create_ep_flux_gif(Fphi, Fz, EP_div, lat_vals.values, z_vals, out_dir, tag)

    scenario_results[tag] = (Fphi, Fz, EP_div, lat_vals.values, z_vals)
    print(f"Finished {tag}")

In [ ]:
# --- Reference long-run processing with deeper debugging ---
# Use full 4D “int.nc” file (with lon=144) instead of merged.nc
ref_path = ("/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
            "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc")
ds_ref = xr.open_dataset(ref_path)
ds_ref = ds_ref.where(ds_ref.time.dt.year == int(year), drop=True)

reference_results = {}
ref_slices = [
    ("long_2.1_6.1", f"{year}-02-01", f"{year}-06-01"),
    ("long_3.1_7.1", f"{year}-03-01", f"{year}-07-01"),
]
for tag, start, end in ref_slices:
    out_dir = os.path.join(target_dir_parent, tag)
    os.makedirs(out_dir, exist_ok=True)
    ds_slice = ds_ref.sel(time=slice(start, end))


    Fphi_r, Fz_r, EP_r, lat_r, z_r = compute_ep_flux_divergence(ds_slice)
    plot_time_mean_ep_flux(Fphi_r, Fz_r, EP_r, lat_r.values, z_r, out_dir, tag)
    create_ep_flux_gif(Fphi_r, Fz_r, EP_r, lat_r.values, z_r, out_dir, tag)

    reference_results[tag] = (Fphi_r, Fz_r, EP_r, lat_r.values, z_r)
    print(f"Finished reference {tag}")

print("\nReference (long run) finished")


In [ ]:
# --- Difference calculations ---
diff_cases = [
    ("Feb_couple_minus_ref",      "0008_Feb_couple",   "long_2.1_6.1"),
    ("Feb_nocouple_minus_ref",    "0008_Feb_nocouple", "long_2.1_6.1"),
    ("Mar_couple_minus_ref",      "0008_Mar_couple",   "long_3.1_7.1"),
    ("Mar_nocouple_minus_ref",    "0008_Mar_nocouple", "long_3.1_7.1"),
    ("Feb_couple_minus_nocouple", "0008_Feb_couple",   "0008_Feb_nocouple"),
    ("Mar_couple_minus_nocouple", "0008_Mar_couple",   "0008_Mar_nocouple"),
]

for diff_tag, A, B in diff_cases:
    out_dir = os.path.join(target_dir_parent, "diffs", diff_tag)
    os.makedirs(out_dir, exist_ok=True)

    # retrieve fields from either scenario_results or reference_results
    if A in scenario_results:
        Fphi_A, Fz_A, EP_A, latA, zA = scenario_results[A]
    else:
        Fphi_A, Fz_A, EP_A, latA, zA = reference_results[A]
    if B in scenario_results:
        Fphi_B, Fz_B, EP_B, latB, zB = scenario_results[B]
    else:
        Fphi_B, Fz_B, EP_B, latB, zB = reference_results[B]

    # compute difference
    Fphi_diff = Fphi_A - Fphi_B
    Fz_diff   = Fz_A   - Fz_B
    EP_diff   = EP_A   - EP_B

    # plot difference
    plot_time_mean_ep_flux(Fphi_diff, Fz_diff, EP_diff, latA, zA, out_dir, diff_tag)
    create_ep_flux_gif(Fphi_diff, Fz_diff, EP_diff, latA, zA, out_dir, diff_tag)

print("EP flux difference analysis complete.")

In [ ]:

# 1. 读 ERA5 并重命名
era5_file = "/home/weiji/restart_exam/era5_data/Era5_uvt_2015to2024.nc"
ds = xr.open_dataset(era5_file).rename({
    'valid_time'    : 'time',
    'pressure_level': 'plev',
    'latitude'      : 'lat',
    'longitude'     : 'lon'
})

# 2. 把 plev 单位从 hPa→Pa
ds = ds.assign_coords(plev = ds['plev'] * 100.0)
ds['plev'].attrs['units'] = 'Pa'

# 3. 只选 DJF
ds = ds.sel(time=ds.time.dt.month.isin([12,1,2]))

# 4. 构造给子程序的最小 Dataset
ds_ep = xr.Dataset({
    'U'   : ds['u'],
    'V'   : ds['v'],
    'T'   : ds['t'],
    'plev': ds['plev'],  # 已经是 Pa 了
    'lat' : ds['lat'],
})

# 5. 调用子程序
Fphi, Fz, EP_div, lat, z = compute_ep_flux_divergence(ds_ep)

# 6. 调用画图函数
outdir = "/home/weiji/restart_exam/plots/epflux_speedtest/ERA5_DJF_validation"
os.makedirs(outdir, exist_ok=True)

plot_time_mean_ep_flux(
    Fphi, Fz, EP_div,
    lat.values, z.values,
    outdir, "ERA5_DJF"
)



print("✅ Done: time‑mean plot saved") 


In [ ]:
def PlotEPfluxArrows(x,y,ep1,ep2,fig,ax,xlim=None,ylim=None,xscale='linear',yscale='linear',invert_y=True, newax=False, pivot='tail',scale=None):
	"""Correctly scales the Eliassen-Palm flux vectors for plotting on a latitude-pressure or latitude-height axis.
		x,y,ep1,ep2 assumed to be xarray.DataArrays.

	INPUTS:
		x       : horizontal coordinate, assumed in degrees (latitude) [degrees]
		y       : vertical coordinate, any units, but usually this is pressure or height
		ep1     : horizontal Eliassen-Palm flux component, in [m2/s2]. Typically, this is ep1_cart from
		           ComputeEPfluxDiv()
		ep2     : vertical Eliassen-Palm flux component, in [U.m/s2], where U is the unit of y.
			       Typically, this is ep2_cart from ComputeEPfluxDiv(), in [hPa.m/s2] and y is pressure [hPa].
		fig     : a matplotlib figure object. This figure contains the axes ax.
		ax      : a matplotlib axes object. This is where the arrows will be plotted onto.
		xlim    : axes limits in x-direction. If None, use [min(x),max(x)]. [None]
		ylim    : axes limits in y-direction. If None, use [min(y),max(y)]. [None]
		xscale  : x-axis scaling. currently only 'linear' is supported. ['linear']
		yscale  : y-axis scaling. 'linear' or 'log' ['linear']
		invert_y: invert y-axis (for pressure coordinates). [True]
		newax   : plot on second y-axis. [False]
		pivot   : keyword argument for quiver() ['tail']
		scale   : keyword argument for quiver() [None]

	OUTPUTS:
	   Fphi*dx : x-component of properly scaled arrows. Units of [m3.inches]
	   Fp*dy   : y-component of properly scaled arrows. Units of [m3.inches]
	   ax   : secondary y-axis if newax == True
	"""
	import numpy as np
	import matplotlib.pyplot as plt
	#
	def Deltas(z,zlim):
		if zlim is None:
			return np.max(z)-np.min(z)
		else:
			return zlim[1]-zlim[0]
	# Scale EP vector components as in Edmon, Hoskins & McIntyre JAS 1980:
	cosphi = np.cos(np.deg2rad(x))
	a0 = 6376000.0 # Earth radius [m]
	grav = 9.81
	# first scaling: Edmon et al (1980), Eqs. 3.1 & 3.13
	Fphi = 2*np.pi/grav*cosphi**2*a0**2*ep1 # [m3.rad]
	Fp   = 2*np.pi/grav*cosphi**2*a0**3*ep2 # [m3.hPa]
	#
	# Now comes what Edmon et al call "distances occupied by 1 radian of
	#  latitude and 1 [hecto]pascal of pressure on the diagram."
	# These distances depend on figure aspect ratio and axis scale
	#
	# first, get the axis width and height for
	#  correct aspect ratio
	width,height = GetAxSize(fig,ax)
	# we use min(),max(), but note that if the actual axis limits
	#  are different, this will be slightly wrong.
	delta_x = Deltas(x,xlim)
	delta_y = Deltas(y,ylim)
	#
	#scale the x-axis:
	if xscale == 'linear':
		dx = width/delta_x/np.pi*180
	else:
		raise ValueError('ONLY LINEAR X-AXIS IS SUPPORTED AT THE MOMENT')
	#scale the y-axis:
	if invert_y:
		y_sign = -1
	else:
		y_sign = 1
	if yscale == 'linear':
		dy = y_sign*height/delta_y
	elif yscale == 'log':
		dy = y_sign*height/y/np.log(np.max(y)/np.min(y))
	#
	# plot the arrows onto axis
	quivArgs = {'angles':'uv','scale_units':'inches','pivot':pivot}
	if scale is not None:
		quivArgs['scale'] = scale
	if newax:
		ax = ax.twinx()
		ax.set_ylabel('pressure [hPa]')
	try:
		ax.quiver(x,y,Fphi*dx,Fp*dy,**quivArgs)
	except:
		ax.quiver(x,y,Fphi.transpose()*dx,Fp.transpose()*dy,**quivArgs)
	if invert_y:
		ax.invert_yaxis()
	if xlim is not None:
		ax.set_xlim(xlim)
	if ylim is not None:
		ax.set_ylim(ylim)
	ax.set_yscale(yscale)
	ax.set_xscale(xscale)
	#
	if newax:
		return Fphi*dx,Fp*dy,ax
	else:
		return Fphi*dx,Fp*dy

def plot_time_mean_ep_flux_scaled(Fphi, Fz, EP_div, lat, z, output_dir, tag):
    """
    在高度-纬度坐标上画出时均 EP 通量与散度，
    EP 矢量经 aostools_functions.PlotEPfluxArrows 自动缩放。
    """
    # 取时间平均
    Fphi_m = Fphi.mean(dim='time')
    Fz_m   = Fz.mean(dim='time')
    EPdiv_m= EP_div.mean(dim='time')

    # 把 DataArray 转成 NumPy array
    Fphi_arr = Fphi_m.values.T    # shape (nlat, nlev)
    Fz_arr   = Fz_m.values.T      # shape (nlat, nlev)



    # 设置网格：纬度 vs 高度（km）
    z_km = z / 1000.0
    LAT2D, Z2D = np.meshgrid(lat, z_km)

    fig, ax = plt.subplots(figsize=(10,6))

    # 先画 EP 散度等值面
    cf = ax.contourf(
        LAT2D, Z2D, EPdiv_m.T,
        levels=np.linspace(-5,5,41), cmap='RdBu_r', extend='both'
    )
    cbar = fig.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label('EP flux divergence (m/s/day)')

    # 再画等值线
    cs = ax.contour(
        LAT2D, Z2D, EPdiv_m.T,
        levels=np.arange(-5,6,1), colors='k', linewidths=0.5
    )
    ax.clabel(cs, fmt='%d', inline=True, fontsize=8)

    # 调用 PlotEPfluxArrows 完成正确的物理缩放
    # xlim 和 ylim 传入轴上显示的纬度/高度范围
    xlim = (lat.min(), lat.max())
    ylim = (z_km.min(), z_km.max())
    # 注意：invert_y=False，因为高度坐标不需要倒置
    Fphi_scaled, Fz_scaled = PlotEPfluxArrows(
        lat, z_km,
        Fphi_arr, Fz_arr,
        fig, ax,
        xlim=xlim, ylim=ylim,
        xscale='linear', invert_y=False,
        pivot='middle', scale=None
    )

    # 用缩放后的矢量画箭头
    # 这里作一个稀疏采样以免箭头太密
    si, sj = 50, 50
    ax.quiver(
        LAT2D[::si,::sj], Z2D[::si,::sj],
        Fphi_scaled.T[::si,::sj], Fz_scaled.T[::si,::sj],
        angles='xy', scale_units='xy', scale=1e25,
        width=0.003, headwidth=4, pivot='middle'
    )

    ax.set_xlim(xlim)
    ax.set_ylim(0, np.max(z_km))
    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Height (km)')
    ax.set_title(f'Time‑mean EP flux & divergence ({tag})')

    os.makedirs(output_dir, exist_ok=True)
    outfile = os.path.join(output_dir, f'EPflux_scaled_{tag}.png')
    fig.tight_layout()
    fig.savefig(outfile, dpi=300)
    plt.show(fig)
    print(f'✅ Saved scaled EP plot: {outfile}')


import aostools_functions as ac
def plot_firststep_ep_flux_ac(ds_hpa, output_dir, tag,
                              step_lat=10, step_plev=2, scale_quiv=1e16):
    """
    单时刻 EP‑flux + divergence  (p‑φ 面, logY, 1000 hPa 在下).
    ds_hpa 必须保持 plev 单位 = hPa.
    """
    # 1) 纬度升序
    ds = ds_hpa.sortby('lat')
    lat_full = ds.lat.values          # 721
    plev_full = ds.plev.values        # 37 (hPa)

    # 2) 取原始数组并计算
    u, v, t = (ds['u'].values, ds['v'].values, ds['t'].values)
    ep1, ep2, div1, div2 = ac.ComputeEPfluxDiv(lat_full, plev_full, u, v, t)
    epdiv = div1 + div2               # (time,plev,lat)

    # 3) 取首时刻 (plev,lat) = (37,721)  —— 注意 **不再转置**
    E1 = ep1[0]                       # (37,721)
    E2 = ep2[0]
    ED = epdiv[0]

    # 4) 抽稀
    lat = lat_full[::step_lat]        # 73
    plev = plev_full[::step_plev]     # 19
    E1 = E1[::step_plev, ::step_lat]  # (19,73)
    E2 = E2[::step_plev, ::step_lat]
    ED = ED[::step_plev, ::step_lat]

    # ---------- 调试输出 ----------
    print(f"DEBUG shapes -> LAT:{lat.shape} PLEV:{plev.shape}  E1:{E1.shape}")
    # ----------

    # 5) 画布
    fig, ax = plt.subplots(figsize=(10,6))

    # 5.1 散度填色
    LAT2D, P2D = np.meshgrid(lat, plev)   # both (19,73)
    cf = ax.contourf(
        LAT2D, P2D, ED,
        levels=np.linspace(-5,5,41),
        cmap='RdBu_r', extend='both'
    )
    fig.colorbar(cf, ax=ax, pad=0.02)\
       .set_label('EP flux divergence (m/s/day)')

    # 5.2 物理缩放 (Edmon+80)
    a0, g = 6.376e6, 9.81
    COS2  = np.cos(np.deg2rad(lat))[None, :]        # (1,73) broadcast→(19,73)
    Fφ = 2*np.pi/g * a0**2 * E1 * COS2              # (19,73)
    Fp = 2*np.pi/g * a0**3 * E2 * COS2

    #   画布 inch 缩放
    W,H = (ax.get_position().width*fig.get_figwidth(),
           ax.get_position().height*fig.get_figheight())
    dx  = W/180.0                                   # inch / deg
    dy  = -H / (plev[:,None] * np.log(plev.max()/plev.min()))  # (19,1)

    U = Fφ * dx
    V = Fp * dy

    ax.quiver(LAT2D, P2D, U, V,
              angles='uv', scale_units='inches',
              scale=scale_quiv, pivot='middle',
              width=0.003, headwidth=4)

    # 6) 轴格式
    ax.set_xlim(-90,90)
    ax.set_yscale('log')
    ax.set_ylim(1000,10)
    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title(f'EP‑flux & divergence at first time step ({tag})')
#    ax.invert_yaxis()

    # 7) 保存
    os.makedirs(output_dir, exist_ok=True)
    outpng = os.path.join(output_dir, f'epflux_first_{tag}.png')
    fig.savefig(outpng, dpi=300, bbox_inches='tight')
    plt.show(fig)
    print("✅ Saved:", outpng)


era5 = "/home/weiji/restart_exam/era5_data/Era5_uvt_2015to2024.nc"
ds0 = (xr.open_dataset(era5)
         .rename({'valid_time':'time','pressure_level':'plev',
                  'latitude':'lat','longitude':'lon'}))

ds_djf = ds0.sel(time=ds0.time.dt.month.isin([12,1,2]))

plot_firststep_ep_flux_ac(
    ds_djf,
    "/home/weiji/restart_exam/plots/epflux_first/ERA5_DJF",
    "ERA5_DJF"
)

In [ ]:
'''
another way of plotting
#!/usr/bin/env python3 
# -*- coding: utf-8 -*-



# --- Constants ---
a = 6.37122e6         # Earth radius (m)
Omega = 7.2921e-5     # Earth rotation rate (rad/s)
g = 9.806             # Gravity (m/s^2)
p0 = 1e5              # Reference pressure (Pa)
H = 7000.0            # Scale height (m)

def compute_ep_flux_divergence(ds):
    """
    Compute Eliassen-Palm (EP) flux components and divergence,
    but with corrected z = -H*ln((plev*100)/p0) so that plev(hPa)->Pa.
    """
    U = ds['U']
    V = ds['V']
    T = ds['T']
    plev = ds['plev']
    lat = ds['lat']

    # potential temperature
    theta = potential_temperature(plev * units.hPa, T * units.K)

    # zonal means and eddies
    u_zm = U.mean(dim='lon')
    v_zm = V.mean(dim='lon')
    theta_zm = theta.mean(dim='lon')
    u_za = U - u_zm
    v_za = V - v_zm
    theta_za = theta - theta_zm

    UVzm = (u_za * v_za).mean(dim='lon')
    VTHETAzm = (v_za * theta_za).mean(dim='lon')

    # --- MODIFIED: convert plev from hPa to Pa for z calculation ---
    p_pa = plev * 100.0        # hPa -> Pa
    z = -H * np.log(p_pa / p0) # now z=0 at p=1000hPa, increases upward

    # background density
    rho0_1d = p0 / (g * H) * np.exp(-z / H)
    rho0 = xr.DataArray(rho0_1d,
                        coords={'plev': plev},
                        dims=('plev',)
                       ).expand_dims(lat=lat)

    # Coriolis and geometry
    phi = np.deg2rad(lat)
    f = 2 * Omega * np.sin(phi)
    cosphi = np.cos(phi)
    acphi = a * cosphi

    # meridional gradient of mean θ
    theta_zm = theta_zm.assign_coords({'z': z})
    dthetaz = theta_zm.differentiate(coord='z', edge_order=2)

    # EP flux components
    Fphi = - rho0 * acphi * UVzm
    Fz   =   f * acphi * rho0 * VTHETAzm / dthetaz

    Fphi = (Fphi
            .assign_coords({'phi': phi})
            .assign_attrs(var_name="meridional EP flux Fφ",
                          units="kg/s²"))
    Fz = (Fz
          .assign_coords({'z': z})
          .assign_attrs(var_name="vertical EP flux Fz",
                        units="kg/s²"))

    # divergence
    div_phi = ((Fphi * cosphi).differentiate('phi', edge_order=2)) / acphi
    div_z   =  Fz.differentiate('z', edge_order=2)
    EP_div = (div_phi + div_z) / (rho0 * acphi) * (3600 * 24)
    EP_div = (EP_div
              .assign_coords({'z': z})
              .assign_attrs(var_name="EP flux divergence",
                            units="m/s/day"))

    return Fphi, Fz, EP_div, lat, z


# === ERA5 DJF 2015–2024 验证 & 绘图，只画时均 ===

# 1. 读取并重命名
era5_file = "/home/weiji/restart_exam/era5_data/Era5_uvt_2015to2024.nc"
ds = xr.open_dataset(era5_file).rename({
    'valid_time'    : 'time',
    'pressure_level': 'plev',
    'latitude'      : 'lat',
    'longitude'     : 'lon'
})

# 2. 选 DJF
ds = ds.sel(time=ds.time.dt.month.isin([12, 1, 2]))

# 3. 最小数据集打包给子程序
ds_ep = xr.Dataset({
    'U'   : ds['u'],
    'V'   : ds['v'],
    'T'   : ds['t'],
    'plev': ds['plev'],
    'lat' : ds['lat'],
})

# 4. 调用
Fphi, Fz, EP_div, lat, z = compute_ep_flux_divergence(ds_ep)

# 5. 画时均
Fphi_m = Fphi.mean(dim='time')
Fz_m   = Fz.mean(dim='time')
EP_m   = EP_div.mean(dim='time')

# 自动设色标为 5–95 百分位，保证底层也能上色
flat = EP_m.values.ravel()
vmin, vmax = np.nanpercentile(flat, [5, 95])
levels = np.linspace(vmin, vmax, 41)

lat2d, z2d = np.meshgrid(lat, z / 1000.0)
fig, ax = plt.subplots(figsize=(10, 6))
cf = ax.contourf(lat2d, z2d, EP_m.T, levels=levels, extend='both')
cb = fig.colorbar(cf, ax=ax, label='EP flux divergence (m/s/day)')
cs = ax.contour(lat2d, z2d, EP_m.T, levels=levels[::5],
                colors='k', linewidths=0.5)
ax.clabel(cs, fmt='%.1f', inline=True, fontsize=8)

# 矢量归一化
s_plev, s_lat = 2, 2
X = lat2d[::s_plev, ::s_lat]
Y = z2d[::s_plev, ::s_lat]
Uq = Fphi_m.values[::s_plev, ::s_lat]
Vq = Fz_m.values[::s_plev, ::s_lat]
mag = np.nanmax(np.hypot(Uq, Vq))
if mag > 0:
    scale = (vmax - vmin) / 2
    Uq = Uq / mag * scale
    Vq = Vq / mag * scale

ax.quiver(X, Y, Uq, Vq,
          angles='xy', scale_units='xy', scale=1,
          width=0.003)

ax.set_ylim(0, np.max(z) / 1000.0)   # 从地面（≈0 km）到 z 的最高处
ax.set_xlabel('Latitude (°)')
ax.set_ylabel('Height (km)')
ax.set_title('Time‑mean QG EP Flux & Divergence (ERA5 DJF 2015–24)')
fig.tight_layout()

outdir = "/home/weiji/restart_exam/plots/epflux_speedtest/ERA5_DJF_validation"
os.makedirs(outdir, exist_ok=True)
fig.savefig(f"{outdir}/time_mean_EPflux_ERA5_DJF_test.png")
plt.close(fig)

print("✅ 完成：已保存 time_mean_EPflux_ERA5_DJF.png")

'''

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import ticker

# --------------------------------------------------------------
def GetAxSize(fig, ax, dpi=False):
    """返回当前轴宽、高（inch；dpi=True → 像素）"""
    bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    w, h = bbox.width, bbox.height
    if dpi:
        w *= fig.dpi
        h *= fig.dpi
    return w, h
# --------------------------------------------------------------

def plot_time_mean_ep_flux_scaled(Fphi, Fz, EP_div,
                                  lat_1d, z_1d,
                                  output_dir, dataset_tag,
                                  s_plev=2, s_lat=2,
                                  arrow_scale=1e16,          # ← 初始全局倍缩，可再调
                                  cmap_div='RdBu_r'):
    """
    Jucker(2021) 缩放：纬度‑高度 (z) 图；高度轴显示 km，箭矢缩放按米。
    """

    # ---------- 0. 时间平均 ----------
    Fphi_m  = Fphi.mean(dim='time')
    Fz_m    = Fz  .mean(dim='time')
    EPdiv_m = EP_div.mean(dim='time')

    print("\n--- time‑mean EP‑flux stats ---")
    print("Fphi_m  min {:.3g}  max {:.3g}".format(float(Fphi_m.min()),
                                                 float(Fphi_m.max())))
    print("Fz_m    min {:.3g}  max {:.3g}".format(float(Fz_m.min()),
                                                 float(Fz_m.max())))

    # ---------- 1. 质量加权 ----------
    a  = 6_371_000.0
    g  = 9.81
    two_pi = 2*np.pi

    cosphi_da = xr.DataArray(np.cos(np.deg2rad(lat_1d)),
                             coords={'lat': Fphi_m['lat']}, dims=('lat',))

    Fphi_hat = (two_pi / g) * a      * cosphi_da * Fphi_m   # [m³ s⁻²]
    Fz_hat   = (two_pi / g) * a**2   * cosphi_da * Fz_m     # [m⁴ s⁻²]

    print("\n--- F_hat stats ---")
    print("Fphi_hat min {:.3g}  max {:.3g}".format(float(Fphi_hat.min()),
                                                  float(Fphi_hat.max())))
    print("Fz_hat   min {:.3g}  max {:.3g}".format(float(Fz_hat.min()),
                                                  float(Fz_hat.max())))

    # ---------- 2. 几何缩放 α,β ----------
    fig, ax = plt.subplots(figsize=(9, 6))

    delta_lat = lat_1d.max() - lat_1d.min()        # [deg]
    delta_z_m = z_1d.max()    - z_1d.min()         # [m]  ← 用米!

    width_in, height_in = GetAxSize(fig, ax)
    dx = width_in  / (delta_lat * np.pi / 180.)    # inch / rad
    dy = height_in /  delta_z_m                    # inch / m

    U_plot = (Fphi_hat * dx).values
    V_plot = (Fz_hat   * dy).values

    mag_in = np.sqrt(U_plot**2 + V_plot**2) / arrow_scale
    print("\n--- arrow length (inch, after scale) ---")
    print("min {:.3g}  max {:.3g}  95‑perc {:.3g}"
          .format(np.nanmin(mag_in), np.nanmax(mag_in),
                  np.nanpercentile(mag_in, 95)))

    # ---------- 3. 底图散度 ----------
    lat2d, z2d = np.meshgrid(lat_1d, z_1d/1000.)   # z→km
    cf = ax.contourf(lat2d, z2d, EPdiv_m.T,
                     levels=np.linspace(-5, 5, 41),
                     cmap=cmap_div, extend='both')
    fig.colorbar(cf, ax=ax, label='EP‑flux divergence (m s⁻¹ day⁻¹)')

    cont = ax.contour(lat2d, z2d, EPdiv_m.T,
                      levels=np.arange(-5, 6, 1),
                      colors='k', linewidths=0.4)
    ax.clabel(cont, fmt='%d', inline=True, fontsize=7)

    # ---------- 4. 画箭矢 ----------
    X  = lat2d[::s_plev, ::s_lat]
    Y  = z2d  [::s_plev, ::s_lat]
    Uq = U_plot[::s_plev, ::s_lat]
    Vq = V_plot[::s_plev, ::s_lat]

    print("\n将绘制箭矢数量：{} × {} = {}".format(Uq.shape[0], Uq.shape[1], Uq.size))

    ax.quiver(X, Y, Uq, Vq,
              angles='uv', scale_units='inches',
              pivot='tail', scale=arrow_scale,
              width=0.003, headwidth=4,
              edgecolor='k', color='none')

    # ---------- 5. 轴 & 保存 ----------
    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Geopotential height (km)')
    ax.set_xlim(lat_1d.min(), lat_1d.max())
    ax.set_ylim(z_1d.min()/1000., z_1d.max()/1000.)
    ax.set_title(f'Time‑mean EP‑flux (Jucker‑scaled) & Divergence\n{dataset_tag}')
    ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
    fig.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    out_png = os.path.join(output_dir,
                           f'time_mean_EPflux_scaled_{dataset_tag}.png')
    fig.savefig(out_png, dpi=300)
    plt.show()

    print("\n图像已保存:", out_png)

# --------------------------------------------------------------


plot_time_mean_ep_flux_scaled(
    Fphi, Fz, EP_div,
    lat.values, z.values,
    outdir, "ERA5_DJF"
)
